### 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")
train_rg3 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3.csv")

### X / Y 분리

In [2]:
TARGET = "PassOrFail"

X = train_cn7.drop(columns=[TARGET])
y = train_cn7[TARGET]

print(X.shape)
print(y.value_counts())
print(y.value_counts(normalize=True))

(1211, 25)
PassOrFail
0    1194
1      17
Name: count, dtype: int64
PassOrFail
0    0.985962
1    0.014038
Name: proportion, dtype: float64


### 상수 컬럼 제거

In [3]:
constant_cols = X.columns[X.nunique() <= 1].tolist()

print("상수 컬럼:", constant_cols)

X = X.drop(columns=constant_cols)

상수 컬럼: ['Clamp_Open_Position']


### 데이터 분리 
> stratify = y

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train")
print(y_train.value_counts())

print("\nTest")
print(y_test.value_counts())

Train
PassOrFail
0    954
1     14
Name: count, dtype: int64

Test
PassOrFail
0    240
1      3
Name: count, dtype: int64


In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Before")
print(y_train.value_counts())

print("\nAfter SMOTE")
print(y_train_smote.value_counts())

Before
PassOrFail
0    954
1     14
Name: count, dtype: int64

After SMOTE
PassOrFail
0    954
1    954
Name: count, dtype: int64


### baseline logistic regression

In [6]:
from sklearn.linear_model import LogisticRegression

lr_smote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = lr_smote.predict(X_test)
y_prob_smote = lr_smote.predict_proba(X_test)[:, 1]

### 평가

In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

print("[CN7 Logistic Regression + SMOTE]")
print("Accuracy :", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_smote, zero_division=0))
print("F1       :", f1_score(y_test, y_pred_smote, zero_division=0))
print("PR-AUC   :", average_precision_score(y_test, y_prob_smote))




[CN7 Logistic Regression + SMOTE]
Accuracy : 0.9547325102880658
Precision: 0.1
Recall   : 0.3333333333333333
F1       : 0.15384615384615385
PR-AUC   : 0.13174603174603175
